# 02 – Modelling: predicting the category from the description

Input: `transaction_description` · Target: `category` · Main metric: macro-F1
Two evaluations: random split (merchants already seen) and merchant split (new merchants).

In [1]:
import pandas as pd

clean = pd.read_parquet("../data/clean_sample.parquet")
print(clean.shape)
clean.head()

(200000, 3)


,transaction_description,merchant,category
0,Exxon - CANADA Store,exxon,Transportation
1,Dunkin' Donuts - USA Branch,dunkin' donuts,Food & Dining
2,Zoo - INDIA Store - Night,zoo,Entertainment & Recreation
3,Lyft - UK Branch,lyft,Transportation
4,Ulta #9601 Store (Cash),ulta,Shopping & Retail


In [2]:
# Random Split
from sklearn.model_selection import train_test_split

train_r, test_r = train_test_split(
    clean, test_size=0.2, random_state=42, stratify=clean["category"]
)
print(len(train_r), len(test_r))

160000 40000


In [3]:
# Merchant split
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(clean, groups=clean["merchant"]))
train_m, test_m = clean.iloc[train_idx], clean.iloc[test_idx]
print(len(train_m), len(test_m))

160485 39515


In [4]:
seen_r = test_r["merchant"].isin(train_r["merchant"]).mean()
seen_m = test_m["merchant"].isin(train_m["merchant"]).mean()
print(f"Test rows with a merchant seen in training - random: {seen_r:.1%}, merchant: {seen_m:.1%}")

Test rows with a merchant seen in training - random: 100.0%, merchant: 0.0%


In [5]:
from sklearn.metrics import f1_score, accuracy_score

def evaluate(model, train, test):
    model.fit(train["transaction_description"], train["category"])
    pred = model.predict(test["transaction_description"])
    return {
        "accuracy": accuracy_score(test["category"], pred),
        "macro_f1": f1_score(test["category"], pred, average="macro"),
    }

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

models = {
    "dummy": make_pipeline(TfidfVectorizer(), DummyClassifier(strategy="most_frequent")),
    "tfidf_word_lr": make_pipeline(
        TfidfVectorizer(analyzer="word", ngram_range=(1, 2)),
        LogisticRegression(max_iter=1000),
    ),
    "tfidf_char_lr": make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True),
        LogisticRegression(max_iter=1000),
    ),
}

In [7]:
results = []
for name, model in models.items():
    for split_name, (tr, te) in {"random": (train_r, test_r), "merchant": (train_m, test_m)}.items():
        scores = evaluate(model, tr, te)
        results.append({"model": name, "split": split_name, **scores})

results_df = pd.DataFrame(results).pivot(index="model", columns="split", values="macro_f1").round(3)
results_df

split,merchant,random
model,,
dummy,0.011,0.018
tfidf_char_lr,0.367,0.986
tfidf_word_lr,0.238,0.986


### First results (macro-F1)

| Model | Random split | Merchant split |
|---|---|---|
| Dummy (most frequent) | 0.018 | 0.011 |
| Word TF-IDF + LR | 0.986 | 0.238 |
| Char TF-IDF + LR | 0.986 | 0.367 |

- **Random split: 0.986** for both TF-IDF models, at the ~98.5% ceiling set by the 7 ambiguous merchants.
  Every test merchant was also in training (100%), so this score measures **memorising ~360 merchant names**,
  not understanding.
- **Merchant split: 0.367 (char) / 0.238 (word).** On merchants it has never seen, the model mostly fails.
  It is still well above chance (~0.10 for 10 categories), so it does learn some real clues.
- **Why so low:** the 4.5M rows contain only ~480 merchants. Repeating each one with random suffixes adds
  no new information, so the model effectively learns from a few hundred examples.
- **Characters beat words on new merchants:** 2–4 letter chunks match parts of unseen names
  ("pizzeria" shares "pizz" with "pizza"), while word features need an exact known word.
- **Conclusion:** a random split would have reported 98.6% and been misleading. The merchant split is the
  honest estimate of how the model will do on a real user's new merchants.

In [8]:
char_model = models["tfidf_char_lr"]
char_model.fit(train_m["transaction_description"], train_m["category"])

test_m = test_m.copy()
test_m["pred"] = char_model.predict(test_m["transaction_description"])
test_m["correct"] = test_m["pred"] == test_m["category"]

per_merchant = (test_m.groupby("merchant")
                .agg(category=("category", "first"),
                     top_pred=("pred", lambda p: p.mode()[0]),
                     accuracy=("correct", "mean"))
                .sort_values("accuracy"))

print(per_merchant.head(20))   # worst merchants
print(per_merchant.tail(20))   # best merchants

                                  category                    top_pred  \
merchant                                                                 
ambulance               Government & Legal  Entertainment & Recreation   
arcade          Entertainment & Recreation           Shopping & Retail   
atf                     Government & Legal        Utilities & Services   
carnival        Entertainment & Recreation        Healthcare & Medical   
burlington               Shopping & Retail              Transportation   
bonus                               Income               Food & Dining   
bloomingdale's           Shopping & Retail               Food & Dining   
bart                        Transportation           Shopping & Retail   
comerica                Financial Services        Utilities & Services   
checkers                     Food & Dining         Charity & Donations   
dollar                      Transportation           Shopping & Retail   
dart                        Transporta

## Error analysis (merchant split, char TF-IDF + LR)

**What it gets right:** names that share a word with merchants seen in training
(`la fitness`, `24 hour fitness`, `pharmacy`, `clinic`, `investment bank`, `electric company`).

**What it gets wrong:**
- **Brands and abbreviations with no clue in the name:** `chase`, `comerica`, `netflix`
- **Generic words that appear in only one merchant:** `festival`, `carnival`, `arcade`, `insurance`.
  The word has meaning, but the model never saw it during training, so knowing what words mean would help.

**All or nothing:** each merchant is either fully right or fully wrong, because the suffixes
(`- UK`, `(Card)`) are random noise and the model decides from the name alone. The merchant-split
score is therefore based on ~90 test merchants, so a single split is unreliable -> GroupKFold next.